# Caso A — Data Augmentation mediante Cruce Cruzado (Cross-Matching)

## 1. Fundamento y Objetivo
En los sistemas de recomendación vocacional basados en perfiles de competencias, generar casos sintéticos con porcentajes arbitrarios (como asignar un 40% inventado) introduce sesgo y falsas correlaciones. 

El **Caso A (Cross-Matching)** resuelve esto utilizando **únicamente la topología real de habilidades de la base de datos**:
- Toma los usuarios ideales (Happy Paths) de cada carrera dentro de un área de conocimiento.
- Evalúa a ese usuario ideal contra **todas las demás carreras de su misma área** (respetando la restricción estricta de área).
- Por ejemplo: Si un usuario tiene el perfil ideal de *Ingeniería en Computación*, al compararse con *Ingeniería en Control y Automatización*, compartirá habilidades de tronco común (cálculo, lógica, física), pero carecerá de las habilidades de especialidad (robótica, instrumentación).
- **Resultado:** Genera casos de coincidencia baja y media (0.0% a 40.0% / 60.0%) de forma 100% natural, fundamentada y libre de etiquetas inventadas.

### Esquema de Identificación:
Cada usuario generado lleva el prefijo de UUID: `CROSS-<uuid>` para trazabilidad en auditoría y particionamiento sin data leakage.

### Celda 1: Importaciones y Conexión

In [1]:
import random
import uuid
from typing import Optional

import polars as pl
import sqlalchemy as sa
from faker import Faker
from IPython.display import display
from pydantic import BaseModel

DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = 'password'
DB_CONECTION = 'athena'

connection_url = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url)
fake = Faker('es_MX')

### Celda 2: Modelos Pydantic

In [2]:
# ── Modelos Pydantic (Validación de Esquema) ─────────────────────────
class Area(BaseModel):
    id_area: int
    nombre_area: str

class Carrera(BaseModel):
    id_carrera: int
    nombre_carrera: str

class Habilidad(BaseModel):
    id_habilidad: int
    descripcion: str

class Usuario(BaseModel):
    id_usuario: Optional[int] = None
    uuid_usuario: str
    nombre: str
    correo: str
    id_area: Optional[int] = None

class Resultado(BaseModel):
    id_resultado: Optional[int] = None
    uuid_usuario: str
    id_carrera: int
    porcentaje_coincidencia: float = 1.0
    top: int = 1

### Celda 3: Helpers de Base de Datos

In [3]:
# ── Funciones Helper de Consulta y Persistencia en BD ──────────────────
def get_all_areas() -> list[dict]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_area, nombre_area
            FROM area
            ORDER BY id_area;
        ''')).mappings().all()
    return [
        {'id_area': row['id_area'], 'nombre_area': row['nombre_area']}
        for row in rows
    ]


def get_all_careers() -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            ORDER BY id_carrera;
        ''')).mappings().all()
    return [
        Carrera(
            id_carrera=row['id_carrera'],
            nombre_carrera=row['nombre_carrera']
        )
        for row in rows
    ]


def get_careers_by_area_select(id_area: int) -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            WHERE id_area = :id_area
            ORDER BY id_carrera;
        '''), {'id_area': id_area}).mappings().all()
    return [
        Carrera(
            id_carrera=row['id_carrera'],
            nombre_carrera=row['nombre_carrera']
        )
        for row in rows
    ]


def get_habilities_by_career(id_carrera: int) -> list[Habilidad]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT h.id_habilidad, h.descripcion
            FROM carrera_habilidad ch
            JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
            WHERE ch.id_carrera = :id_carrera
            ORDER BY h.id_habilidad;
        '''), {'id_carrera': id_carrera}).mappings().all()
    return [
        Habilidad(
            id_habilidad=row['id_habilidad'],
            descripcion=row['descripcion']
        )
        for row in rows
    ]


def generate_name_and_email() -> tuple[str, str]:
    name = fake.name()
    fake_email = fake.email()
    email_local, email_domain = fake_email.split('@', 1)
    email = f'{email_local}_{uuid.uuid4().hex[:10]}@{email_domain}'
    return name, email


def submit_user(user: Usuario) -> Usuario:
    with engine.begin() as conn:
        row = conn.execute(sa.text('''
            INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
            VALUES (:nombre, :correo, :uuid_usuario, :id_area)
            RETURNING id_usuario, uuid_usuario;
        '''), {
            'nombre': user.nombre,
            'correo': user.correo,
            'uuid_usuario': user.uuid_usuario,
            'id_area': user.id_area
        }).mappings().one()
    return Usuario(
        id_usuario=row['id_usuario'],
        uuid_usuario=str(row['uuid_usuario']),
        nombre=user.nombre,
        correo=user.correo,
        id_area=user.id_area,
    )


def construct_result(
    uuid_usuario: str,
    id_carrera: int,
    porcentaje_coincidencia: float = 1.0,
    top: int = 1
) -> Resultado:
    with engine.begin() as conn:
        conn.execute(sa.text('''
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        '''), {
            'uuid_usuario': uuid_usuario,
            'id_carrera': id_carrera,
            'porcentaje_coincidencia': porcentaje_coincidencia,
            'top': top,
        })
    return Resultado(
        uuid_usuario=uuid_usuario,
        id_carrera=id_carrera,
        porcentaje_coincidencia=porcentaje_coincidencia,
        top=top,
    )


def get_happy_path_users_by_area(id_area: int) -> list[dict]:
    query = '''
        SELECT
            u.id_usuario,
            u.uuid_usuario,
            u.nombre,
            u.correo,
            u.id_area,
            uh.id_habilidad,
            MAX(uh.cumplimiento_criterio) AS cumplimiento_criterio
        FROM usuario u
        JOIN usuario_habilidad uh ON uh.id_usuario = u.id_usuario
        WHERE u.id_area = :id_area
          AND u.uuid_usuario LIKE 'USER-%'
        GROUP BY u.id_usuario, u.uuid_usuario, u.nombre,
                 u.correo, u.id_area, uh.id_habilidad
        ORDER BY u.id_usuario, uh.id_habilidad;
    '''
    with engine.begin() as conn:
        rows = conn.execute(
            sa.text(query),
            {'id_area': id_area}
        ).mappings().all()
    usuarios = {}
    for row in rows:
        uid = row['id_usuario']
        if uid not in usuarios:
            usuarios[uid] = {
                'id_usuario': uid,
                'uuid_usuario': row['uuid_usuario'],
                'nombre': row['nombre'],
                'correo': row['correo'],
                'id_area': row['id_area'],
                'habilidades': {}
            }
        usuarios[uid]['habilidades'][row['id_habilidad']] = float(
            row['cumplimiento_criterio']
        )
    return list(usuarios.values())

### Celda 4: Algoritmo de Cross-Matching e Inserción
Itera sobre cada área, extrae los perfiles Happy Path existentes, calcula la intersección estricta de habilidades requeridas y genera el nuevo registro vocacional en `resultado`.

In [4]:
def run_cross_matching(persist_to_db: bool = False) -> list[dict]:
    '''
    Calcula el cruce en memoria y balancea los casos entre las tres areas.
    Si se solicita, persiste el lote completo en una sola transaccion.
    '''
    with engine.connect() as conn:
        career_rows = conn.execute(sa.text('''
            SELECT c.id_carrera, c.nombre_carrera, c.id_area, ch.id_habilidad
            FROM carrera c
            JOIN carrera_habilidad ch ON ch.id_carrera = c.id_carrera
            ORDER BY c.id_area, c.id_carrera, ch.id_habilidad;
        ''')).mappings().all()
        user_skill_rows = conn.execute(sa.text('''
            SELECT u.id_usuario, u.id_area, uh.id_habilidad,
                   MAX(uh.cumplimiento_criterio) AS cumplimiento_criterio
            FROM usuario u
            JOIN usuario_habilidad uh ON uh.id_usuario = u.id_usuario
            WHERE u.uuid_usuario LIKE 'USER-%'
            GROUP BY u.id_usuario, u.id_area, uh.id_habilidad
            ORDER BY u.id_usuario, uh.id_habilidad;
        ''')).mappings().all()

    careers_by_area = {}
    for row in career_rows:
        career = careers_by_area.setdefault(row['id_area'], {}).setdefault(
            row['id_carrera'],
            {
                'id_carrera': row['id_carrera'],
                'nombre_carrera': row['nombre_carrera'],
                'habilidades': set()
            }
        )
        career['habilidades'].add(row['id_habilidad'])

    users = {}
    for row in user_skill_rows:
        user = users.setdefault(
            row['id_usuario'],
            {
                'id_usuario': row['id_usuario'],
                'id_area': row['id_area'],
                'habilidades': {}
            }
        )
        user['habilidades'][row['id_habilidad']] = float(row['cumplimiento_criterio'])

    resumen = []
    pending = []
    for user in users.values():
        for career in careers_by_area.get(user['id_area'], {}).values():
            career_skills = career['habilidades']
            common_skills = career_skills.intersection(user['habilidades'])
            total_required = len(career_skills)
            completion_sum = sum(
                user['habilidades'][skill_id]
                for skill_id in common_skills
            )
            percentage = (
                round(completion_sum / total_required, 4)
                if total_required else 0.0
            )

            if percentage >= 1.0:
                continue

            cross_uuid = f'CROSS-{uuid.uuid4()}'
            fake_name = fake.name()
            fake_email = fake.email()
            email_local, email_domain = fake_email.split('@', 1)
            email = f'{email_local}_{uuid.uuid4().hex[:10]}@{email_domain}'
            summary_index = len(resumen)
            resumen.append({
                'id_usuario': None,
                'uuid_usuario': cross_uuid,
                'id_area': user['id_area'],
                'id_carrera': career['id_carrera'],
                'carrera_evaluada': career['nombre_carrera'],
                'total_requeridas': total_required,
                'habilidades_comunes': len(common_skills),
                'suma_cumplimiento': round(completion_sum, 2),
                'porcentaje_coincidencia': percentage
            })
            pending.append({
                'summary_index': summary_index,
                'uuid_usuario': cross_uuid,
                'nombre': fake_name,
                'correo': email,
                'id_area': user['id_area'],
                'habilidades': user['habilidades'],
                'id_carrera': career['id_carrera'],
                'porcentaje_coincidencia': percentage
            })

    pending_by_area = {}
    for item in pending:
        pending_by_area.setdefault(item['id_area'], []).append(item)

    if len(pending_by_area) < 3:
        raise ValueError('No hay candidatos en las tres areas para balancear el lote.')

    target_per_area = min(
        len(area_items)
        for area_items in pending_by_area.values()
    )
    balanced_pending = []
    balanced_resumen = []
    for area_id in sorted(pending_by_area):
        for item in pending_by_area[area_id][:target_per_area]:
            original_summary_index = item['summary_index']
            item['summary_index'] = len(balanced_resumen)
            balanced_resumen.append(resumen[original_summary_index])
            balanced_pending.append(item)
    pending = balanced_pending
    resumen = balanced_resumen

    if persist_to_db and pending:
        with engine.begin() as conn:
            conn.execute(sa.text('''
                INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
                VALUES (:nombre, :correo, :uuid_usuario, :id_area);
            '''), [
                {
                    'nombre': item['nombre'],
                    'correo': item['correo'],
                    'uuid_usuario': item['uuid_usuario'],
                    'id_area': item['id_area']
                }
                for item in pending
            ])

            inserted_users = conn.execute(sa.text('''
                SELECT id_usuario, uuid_usuario
                FROM usuario
                WHERE uuid_usuario = ANY(:uuids);
            '''), {
                'uuids': [item['uuid_usuario'] for item in pending]
            }).mappings().all()
            ids_by_uuid = {
                row['uuid_usuario']: row['id_usuario']
                for row in inserted_users
            }
            for item in pending:
                item['id_usuario'] = ids_by_uuid[item['uuid_usuario']]
                resumen[item['summary_index']]['id_usuario'] = item['id_usuario']

            skill_rows = [
                {
                    'id_usuario': item['id_usuario'],
                    'id_habilidad': skill_id,
                    'cumplimiento_criterio': completion
                }
                for item in pending
                for skill_id, completion in item['habilidades'].items()
            ]
            conn.execute(sa.text('''
                INSERT INTO usuario_habilidad
                    (id_usuario, id_habilidad, cumplimiento_criterio)
                VALUES (:id_usuario, :id_habilidad, :cumplimiento_criterio);
            '''), skill_rows)

            result_rows = [
                {
                    'uuid_usuario': item['uuid_usuario'],
                    'id_carrera': item['id_carrera'],
                    'porcentaje_coincidencia': item['porcentaje_coincidencia'],
                    'top': 1
                }
                for item in pending
            ]
            conn.execute(sa.text('''
                INSERT INTO resultado
                    (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
                VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
                ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                    porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                    top = EXCLUDED.top;
            '''), result_rows)
    elif not persist_to_db:
        for index, item in enumerate(resumen, start=1):
            item['id_usuario'] = f'SIM-{index}'

    return resumen

# Inserta la misma cantidad de usuarios CROSS en cada area.
resumen_cross = run_cross_matching(persist_to_db=True)
df_cross = pl.DataFrame(resumen_cross)
print(f'Casos Caso A insertados: {df_cross.height}')
print(df_cross.group_by('id_area').len().sort('id_area'))
print(f'Promedio de coincidencia: {df_cross["porcentaje_coincidencia"].mean():.4f}')
print(f'Maximo porcentaje cruzado: {df_cross["porcentaje_coincidencia"].max():.4f}')
display(df_cross.head(8))

Casos Caso A insertados: 330
shape: (3, 2)
┌─────────┬─────┐
│ id_area ┆ len │
│ ---     ┆ --- │
│ i64     ┆ u32 │
╞═════════╪═════╡
│ 1       ┆ 110 │
│ 2       ┆ 110 │
│ 3       ┆ 110 │
└─────────┴─────┘
Promedio de coincidencia: 0.0715
Maximo porcentaje cruzado: 0.4000


id_usuario,uuid_usuario,id_area,id_carrera,carrera_evaluada,total_requeridas,habilidades_comunes,suma_cumplimiento,porcentaje_coincidencia
i64,str,i64,i64,str,i64,i64,f64,f64
67,"""CROSS-050f086d-a1ed-4691-8842-…",1,2,"""Ingeniería Ambiental""",6,1,1.0,0.1667
68,"""CROSS-03414dd3-b786-4f40-b0b4-…",1,3,"""Ingeniería Biomédica""",6,1,1.0,0.1667
69,"""CROSS-a6ebd064-258d-4981-8c97-…",1,4,"""Ingeniería Biónica""",6,0,0.0,0.0
70,"""CROSS-250da461-63be-4d98-9a0c-…",1,5,"""Ingeniería Bioquímica""",6,0,0.0,0.0
71,"""CROSS-642320bc-771c-4f57-b9e2-…",1,6,"""Ingeniería Biotecnológica""",5,0,0.0,0.0
72,"""CROSS-e3dc528f-d567-4f60-881e-…",1,7,"""Ingeniería Civil""",5,1,1.0,0.2
73,"""CROSS-3c16ae00-8765-441e-8c0a-…",1,8,"""Ingeniería Eléctrica""",5,0,0.0,0.0
74,"""CROSS-5b77b054-35e0-41f2-a381-…",1,9,"""Ingeniería en Alimentos""",5,0,0.0,0.0


### Celda 5: Distribución Estadística y Visualización del Caso A
Observamos cómo la intersección natural distribuye los casos en deciles bajos (0.0 a 0.40), reflejando que la gran mayoría de carreras de un área comparten pocas o ninguna habilidad de alta especialidad.

In [5]:
dist_a = df_cross.with_columns(
    ((pl.col('porcentaje_coincidencia') * 10).floor() / 10.0).alias('decil')
).group_by('decil').len().sort('decil')

print('=== DISTRIBUCIÓN PORCENTUAL CASO A ===')
total = df_cross.height
for row in dist_a.iter_rows(named=True):
    pct = (row['len'] / total) * 100
    barra = '█' * int(pct // 2)
    print(f"{row['decil']:.1f} - {row['decil']+0.1:.1f}: {row['len']:4d} casos ({pct:5.1f}%) | {barra}")

=== DISTRIBUCIÓN PORCENTUAL CASO A ===
0.0 - 0.1:  212 casos ( 64.2%) | ████████████████████████████████
0.1 - 0.2:    5 casos (  1.5%) | 
0.2 - 0.3:  112 casos ( 33.9%) | ████████████████
0.4 - 0.5:    1 casos (  0.3%) | 
